# Responsible AI & the Human Side

Companion notebook for the [Responsible AI lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/13-responsible-ai-and-the-human-side).

**The idea in one sentence.** Fairness is **not one number** — different fairness metrics
(demographic parity vs equal opportunity) can be mathematically **incompatible**;
**differential privacy** trades an $\varepsilon$ budget for accuracy; and naive top-1
recommenders **collapse** into feedback loops without exploration.

We build fairness metrics, DP, and a feedback-loop simulation from scratch, and **validate
the DP privacy–accuracy trade-off and feedback-loop collapse**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. Fairness metrics: demographic parity vs equal opportunity vs predictive parity

Two groups $A$ and $B$ apply for a loan. The ground-truth label $Y$ is 1 if they would repay. Base rates differ across groups (group $B$ is over-represented in the high-risk tail of the training distribution, not because of anything fundamental — sampling and historical-label bias). The classifier outputs $\hat{y}$.

We compute three parities:

- **Demographic parity:** $P(\hat{y}=1 \mid A=a) = P(\hat{y}=1 \mid A=b)$ — same approval rate across groups.
- **Equal opportunity:** $P(\hat{y}=1 \mid Y=1, A=a) = P(\hat{y}=1 \mid Y=1, A=b)$ — same TPR across groups.
- **Predictive parity:** $P(Y=1 \mid \hat{y}=1, A=a) = P(Y=1 \mid \hat{y}=1, A=b)$ — same PPV across groups.

When base rates differ, you can satisfy at most one of these exactly.

In [ ]:
def simulate_population(n_per_group=5000, base_rate_a=0.65, base_rate_b=0.45,
                       noise_a=0.18, noise_b=0.26, seed=0):
    """Generate (Y, score, group) for two groups with different base rates and noise levels.

    Group A: cleaner signal (lower noise), higher base rate (richer historical data).
    Group B: noisier signal, lower base rate.
    This mirrors realistic sampling/label bias — not malice, just messier data on group B.
    """
    rng_local = np.random.default_rng(seed)
    n = n_per_group
    y_a = (rng_local.random(n) < base_rate_a).astype(int)
    y_b = (rng_local.random(n) < base_rate_b).astype(int)
    # Score = signal + noise. Higher score -> more likely positive.
    score_a = y_a + rng_local.normal(0, noise_a, size=n)
    score_b = y_b + rng_local.normal(0, noise_b, size=n)
    y = np.concatenate([y_a, y_b])
    score = np.concatenate([score_a, score_b])
    group = np.array(['A'] * n + ['B'] * n)
    return y, score, group


def parity_metrics(y, yhat, group):
    """Compute the three parity metrics across groups A and B."""
    out = {}
    for g in ('A', 'B'):
        m = group == g
        positive_rate = yhat[m].mean()
        true_positive_mask = m & (y == 1)
        tpr = yhat[true_positive_mask].mean() if true_positive_mask.any() else float('nan')
        predicted_positive_mask = m & (yhat == 1)
        ppv = y[predicted_positive_mask].mean() if predicted_positive_mask.any() else float('nan')
        out[g] = {'positive_rate': positive_rate, 'tpr': tpr, 'ppv': ppv}
    return out


y, score, group = simulate_population()

# A single global threshold — the default that fails fairness.
threshold = 0.5
yhat = (score > threshold).astype(int)
metrics = parity_metrics(y, yhat, group)

print('Single global threshold = 0.5')
print(f"  Group A: approval rate = {metrics['A']['positive_rate']:.3f}, "
      f"TPR = {metrics['A']['tpr']:.3f}, PPV = {metrics['A']['ppv']:.3f}")
print(f"  Group B: approval rate = {metrics['B']['positive_rate']:.3f}, "
      f"TPR = {metrics['B']['tpr']:.3f}, PPV = {metrics['B']['ppv']:.3f}")
print(f"\nDemographic parity gap (|approval_A - approval_B|): "
      f"{abs(metrics['A']['positive_rate'] - metrics['B']['positive_rate']):.3f}")
print(f"Equal-opportunity gap (|TPR_A - TPR_B|):          "
      f"{abs(metrics['A']['tpr'] - metrics['B']['tpr']):.3f}")
print(f"Predictive-parity gap (|PPV_A - PPV_B|):           "
      f"{abs(metrics['A']['ppv'] - metrics['B']['ppv']):.3f}")

In [ ]:
# Try to fix equal opportunity by using a per-group threshold (one of the standard mitigations).
def threshold_for_target_tpr(y, score, target_tpr=0.85):
    pos_scores = np.sort(score[y == 1])
    # Pick the threshold where TPR = target.
    idx = int((1 - target_tpr) * len(pos_scores))
    idx = max(0, min(idx, len(pos_scores) - 1))
    return pos_scores[idx]


t_a = threshold_for_target_tpr(y[group == 'A'], score[group == 'A'], target_tpr=0.85)
t_b = threshold_for_target_tpr(y[group == 'B'], score[group == 'B'], target_tpr=0.85)
yhat_eo = np.where(group == 'A', score > t_a, score > t_b).astype(int)
metrics_eo = parity_metrics(y, yhat_eo, group)

# Plot side-by-side: global threshold vs per-group thresholds tuned for equal opportunity.
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for ax, label, m in zip(axes, ('Global threshold = 0.5', 'Per-group threshold (target TPR = 0.85)'),
                        (metrics, metrics_eo)):
    metrics_x = ['approval rate', 'TPR', 'PPV']
    a_vals = [m['A']['positive_rate'], m['A']['tpr'], m['A']['ppv']]
    b_vals = [m['B']['positive_rate'], m['B']['tpr'], m['B']['ppv']]
    x = np.arange(3)
    w = 0.36
    ax.bar(x - w / 2, a_vals, w, color=BRAND, label='group A')
    ax.bar(x + w / 2, b_vals, w, color=ROSE, label='group B')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_x)
    ax.set_ylim(0, 1.05)
    ax.set_title(label)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('Per-group thresholds: A = {:.3f}, B = {:.3f}'.format(t_a, t_b))
print(f"  Group A: approval = {metrics_eo['A']['positive_rate']:.3f}, "
      f"TPR = {metrics_eo['A']['tpr']:.3f}, PPV = {metrics_eo['A']['ppv']:.3f}")
print(f"  Group B: approval = {metrics_eo['B']['positive_rate']:.3f}, "
      f"TPR = {metrics_eo['B']['tpr']:.3f}, PPV = {metrics_eo['B']['ppv']:.3f}")
print(f"\nEqual-opportunity gap fell from "
      f"{abs(metrics['A']['tpr'] - metrics['B']['tpr']):.3f} to "
      f"{abs(metrics_eo['A']['tpr'] - metrics_eo['B']['tpr']):.3f}.")
print(f"Predictive-parity gap moved to "
      f"{abs(metrics_eo['A']['ppv'] - metrics_eo['B']['ppv']):.3f} — trading one fairness for another.")

What the numbers make obvious:

- The global threshold gives group A higher TPR *and* higher PPV — the classic outcome when one group has a cleaner signal.
- Tuning per-group thresholds closes the equal-opportunity gap by accepting more borderline cases in group B. The PPV gap widens in the other direction as a consequence.
- This is the impossibility result in miniature: when base rates differ, fixing one parity perturbs the others. Choose the parity definition that matches the application before you optimise.

## 2. Calibration gap: aggregate calibration can hide per-group miscalibration

A reliability diagram bins predicted probabilities and plots the *empirical positive rate* in each bin. A perfectly calibrated model lies on the diagonal — predicted 0.7 means "actually 70% positive". We turn the classifier scores into pseudo-probabilities with a sigmoid and compare reliability curves per group.

In [ ]:
def reliability_curve(y_true, prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    centres = 0.5 * (bins[:-1] + bins[1:])
    empirical = np.full(n_bins, np.nan)
    counts = np.zeros(n_bins, dtype=int)
    for i in range(n_bins):
        m = (prob >= bins[i]) & (prob < bins[i + 1] if i < n_bins - 1 else prob <= bins[i + 1])
        if m.any():
            empirical[i] = y_true[m].mean()
            counts[i] = int(m.sum())
    return centres, empirical, counts


# Pseudo-probability via sigmoid; small temperature gap between groups bakes in a calibration gap.
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

prob_a = sigmoid(2.2 * score[group == 'A'] - 1.0)
prob_b = sigmoid(1.5 * score[group == 'B'] - 1.0)
y_a = y[group == 'A']
y_b = y[group == 'B']

centres_a, emp_a, cnt_a = reliability_curve(y_a, prob_a)
centres_b, emp_b, cnt_b = reliability_curve(y_b, prob_b)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot([0, 1], [0, 1], color='#94a3b8', lw=1, linestyle='--', alpha=0.6, label='perfectly calibrated')
ax.plot(centres_a, emp_a, 'o-', color=BRAND, lw=2, markersize=7, label='group A')
ax.plot(centres_b, emp_b, 'o-', color=ROSE, lw=2, markersize=7, label='group B')
ax.set_xlabel('predicted probability')
ax.set_ylabel('empirical positive rate')
ax.set_title('Reliability diagram by group — group B is systematically under-confident')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Aggregate calibration: average the curves. Can mask per-group gaps.
prob_all = np.concatenate([prob_a, prob_b])
y_all = np.concatenate([y_a, y_b])
centres_all, emp_all, _ = reliability_curve(y_all, prob_all)
print('Aggregate reliability curve:')
for c, e in zip(centres_all, emp_all):
    print(f'  predicted {c:.2f} -> empirical {e:.3f}')
print('\nAggregate curve looks OK; per-group curves above tell the real story.')

Two lessons from the gap:

- The aggregate reliability curve can look acceptably calibrated while one group's curve sits well off the diagonal. *Always* break down calibration by group when the model touches users.
- A miscalibrated confidence display is worse than no confidence display — it teaches users that the number lies. Pair every UX confidence indicator with per-group calibration in monitoring.

## 3. Differential privacy: the epsilon-accuracy trade-off

We compute a sum statistic on a dataset and release it with epsilon-differential privacy by adding Laplace noise of scale $1/\epsilon$ (the sensitivity of a sum over bounded contributions is 1 when each individual contributes at most 1). Smaller epsilon means stronger privacy; the noise drowns the signal at very small epsilon.

We sweep epsilon and plot mean absolute error vs the true sum. The shape is the universal privacy/utility curve.

In [ ]:
def dp_sum(values, epsilon, sensitivity=1.0, rng_local=None):
    """Release sum(values) with epsilon-DP via Laplace noise of scale sensitivity / epsilon."""
    if rng_local is None:
        rng_local = np.random.default_rng()
    noise = rng_local.laplace(loc=0.0, scale=sensitivity / epsilon)
    return values.sum() + noise


# Bounded individual contributions in [0, 1] — sensitivity = 1.
data = rng.uniform(0, 1, size=10000)
true_sum = data.sum()

epsilons = np.array([0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0])
n_trials = 400

mae = np.empty_like(epsilons)
p95 = np.empty_like(epsilons)
for i, eps in enumerate(epsilons):
    rng_inner = np.random.default_rng(100 + i)
    releases = np.array([dp_sum(data, eps, rng_local=rng_inner) for _ in range(n_trials)])
    err = np.abs(releases - true_sum)
    mae[i] = err.mean()
    p95[i] = np.percentile(err, 95)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(epsilons, mae, 'o-', color=BRAND, lw=2, markersize=7, label='mean abs error')
ax.plot(epsilons, p95, 's--', color=ROSE, lw=2, markersize=7, label='95th-pct abs error')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('privacy budget $\\epsilon$ (smaller = more private)')
ax.set_ylabel('absolute error in released sum')
ax.set_title('DP sum: noise of scale $1/\\epsilon$ — strong privacy costs accuracy')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'True sum: {true_sum:.2f}')
for eps, m, p in zip(epsilons, mae, p95):
    print(f'  epsilon = {eps:6.2f} -> mean abs error = {m:7.3f}, p95 = {p:7.3f}')

### Validate: differential privacy trades an $\varepsilon$ budget for accuracy

DP releases a statistic with Laplace noise of scale $\text{sensitivity}/\varepsilon$. A
*smaller* $\varepsilon$ (stronger privacy) means *more* noise and a *less accurate* release.
We confirm the mean absolute error falls as $\varepsilon$ grows — the fundamental
privacy–accuracy trade-off.

In [ ]:
for eps in [0.01, 0.1, 1.0, 10.0]:
    err = np.mean([abs(dp_sum(data, eps, rng_local=np.random.default_rng(s)) - true_sum) for s in range(400)])
    print(f'epsilon={eps:5.2f} (privacy {"strong" if eps<0.5 else "weak"}): mean |error| = {err:.2f}')
e_small = np.mean([abs(dp_sum(data, 0.01, rng_local=np.random.default_rng(s)) - true_sum) for s in range(400)])
e_large = np.mean([abs(dp_sum(data, 10.0, rng_local=np.random.default_rng(s)) - true_sum) for s in range(400)])
assert e_small > e_large, 'smaller epsilon (more privacy) means more noise / error'
print('\n✅ the privacy–accuracy trade-off: stronger privacy (smaller epsilon) costs accuracy')

Three operational implications:

- At epsilon = 0.01 (very strong privacy) the noise dwarfs the signal — the released sum is useless for any decision. At epsilon = 10 the noise is invisible but the privacy guarantee is weak.
- Real DP deployments live in the middle (typically epsilon between 0.5 and 5 for a single release, with a tracked global budget). Pick the operating point with stakeholders before instrumenting.
- The same shape governs DP-SGD, federated learning with DP, and any release mechanism — only the constants change.

## 4. Feedback-loop collapse in a top-1 recommender

A recommender that only ever shows the top-1 item to each user creates a closed loop: only the shown items get clicked, only clicks become training data, and the long tail dies. We simulate 50 items with stable true relevances and a recommender that retrains on observed clicks each round. Diversity (the fraction of items that ever get shown) collapses fast.

Mitigation: inject diversity at ranking time. We compare top-1 with an epsilon-greedy policy (90% top-1, 10% random) — a one-line change that keeps the tail alive.

In [ ]:
def simulate_recommender(rounds=200, n_items=50, users_per_round=200, epsilon_explore=0.0, seed=7):
    """Each round, recommend one item per user, observe noisy clicks, refit a relevance estimate.

    epsilon_explore = 0.0 -> pure top-1 (no exploration).
    epsilon_explore = 0.1 -> 10 percent of users get a uniform-random item.
    """
    rng_local = np.random.default_rng(seed)
    true_relevance = rng_local.uniform(0.1, 0.9, size=n_items)
    shown_count = np.zeros(n_items, dtype=int)
    click_count = np.zeros(n_items, dtype=int)
    # Start with a noisy estimate biased toward a few items (warm-start bias).
    estimate = true_relevance + rng_local.normal(0, 0.25, size=n_items)
    estimate[:5] += 0.5   # five items launch ahead due to historical warm-start.

    coverage_over_time = []
    for _ in range(rounds):
        explore_mask = rng_local.random(users_per_round) < epsilon_explore
        top_item = int(np.argmax(estimate))
        for u in range(users_per_round):
            item = int(rng_local.integers(n_items)) if explore_mask[u] else top_item
            shown_count[item] += 1
            clicked = int(rng_local.random() < true_relevance[item])
            click_count[item] += clicked
        # Refit: estimate = empirical click rate (only known for shown items).
        with np.errstate(divide='ignore', invalid='ignore'):
            empirical = np.where(shown_count > 0, click_count / np.maximum(shown_count, 1), 0.0)
        # Items never shown keep an old (over-confident-zero) estimate -> the collapse mechanism.
        estimate = np.where(shown_count > 0, empirical, estimate)
        coverage_over_time.append(int((shown_count > 0).sum()))
    return np.array(coverage_over_time), shown_count, true_relevance


cov_greedy, shown_greedy, true_rel = simulate_recommender(epsilon_explore=0.0, seed=7)
cov_eps,    shown_eps,    _        = simulate_recommender(epsilon_explore=0.10, seed=7)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(cov_greedy, color=ROSE, lw=2, label='top-1 only (no exploration)')
axes[0].plot(cov_eps, color=TEAL, lw=2, label='epsilon-greedy (10 percent random)')
axes[0].set_xlabel('round')
axes[0].set_ylabel('# distinct items ever shown')
axes[0].set_title('Diversity collapses without exploration')
axes[0].set_ylim(0, 52)
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

items = np.arange(len(true_rel))
order = np.argsort(-true_rel)
axes[1].bar(items, shown_greedy[order], color=ROSE, alpha=0.85, label='top-1 only')
axes[1].bar(items, shown_eps[order], color=TEAL, alpha=0.6, label='epsilon-greedy')
axes[1].set_xlabel('items, sorted by true relevance (best -> worst)')
axes[1].set_ylabel('# times shown')
axes[1].set_title('Top-1 monopolises shows; epsilon-greedy keeps the tail alive')
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'After all rounds — coverage:')
print(f'  top-1 only:       {cov_greedy[-1]:2d} of {len(true_rel)} items ever shown')
print(f'  epsilon-greedy:   {cov_eps[-1]:2d} of {len(true_rel)} items ever shown')

### Validate: a top-1 recommender collapses without exploration

A pure top-1 recommender only ever shows the currently-highest-estimated items, so items it
underrates never get a chance to prove themselves — the catalogue **collapses** to a few
items. A little exploration keeps more of the catalogue alive. We compare catalogue coverage
with and without exploration.

In [ ]:
def coverage(eps_explore):
    # simulate_recommender returns (coverage_over_time, shown_count, true_relevance)
    cov_series, shown_count, _ = simulate_recommender(epsilon_explore=eps_explore)
    return int((shown_count > 0).sum())   # distinct items ever shown
cov_greedy = coverage(0.0)
cov_explore = coverage(0.1)
print(f'catalogue items surfaced -- pure top-1: {cov_greedy}, with 10% exploration: {cov_explore}')
assert cov_explore >= cov_greedy, 'exploration surfaces at least as much of the catalogue'
print('\n✅ pure top-1 collapses the catalogue; exploration keeps items alive (breaks the feedback loop)')

What the simulation shows:

- The pure top-1 policy collapses to a handful of items in a few rounds. Items that lose the initial warm-start lottery are *never* shown again — their estimate stays at the warm-start value, no clicks accumulate, the model never has a reason to surface them.
- A 10% random-exploration policy is a one-line change. It keeps every item being shown, lets new items climb when they deserve to, and barely costs the top-line click rate.
- Real recommenders use richer mitigations (Thompson sampling, MMR, determinantal point processes) but the structural point is the same: *some* diversity injection is mandatory whenever your model's outputs become its future training data.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **"fairness" is not one metric** | demographic parity & equal opportunity can't both hold at different base rates (demo) |
| **privacy costs accuracy** | smaller $\varepsilon$ = more noise = less accurate (verified) |
| **feedback loops** | pure top-1 collapses the catalogue (verified); add exploration |
| **calibration across groups** | a globally-calibrated model can be miscalibrated per group |
| **proxy variables** | dropping a protected attribute doesn't remove bias if proxies remain |

Demo: demographic parity and equal opportunity trade off at different base rates.

In [ ]:
# Fairness metrics can be INCOMPATIBLE: demographic parity (equal positive rates across
# groups) and equal opportunity (equal true-positive rates) generally cannot BOTH hold when
# the groups have different base rates. We show both gaps are non-zero under one threshold.
m = parity_metrics(y, yhat, group)
dp_gap = abs(m['A']['positive_rate'] - m['B']['positive_rate'])  # demographic parity
eo_gap = abs(m['A']['tpr'] - m['B']['tpr'])                      # equal opportunity
print(f'demographic-parity gap (|approval_A - approval_B|): {dp_gap:.3f}')
print(f'equal-opportunity gap  (|TPR_A - TPR_B|)          : {eo_gap:.3f}')
assert dp_gap > 0 and eo_gap > 0, 'with different base rates a single threshold leaves both gaps open'
print('\nForcing one gap to 0 generally moves the other away from 0 -> there is no single "fair".')
print('You must choose which fairness criterion your context requires.')

## Your turn — implement the three parity metrics

Your goal: implement `parity_gap(y, yhat, group, kind)` that returns the absolute difference between groups A and B for the chosen parity:

- `'demographic'` — $|P(\hat{y}=1 \mid A) - P(\hat{y}=1 \mid B)|$
- `'equal_opportunity'` — $|P(\hat{y}=1 \mid Y=1, A) - P(\hat{y}=1 \mid Y=1, B)|$
- `'predictive_parity'` — $|P(Y=1 \mid \hat{y}=1, A) - P(Y=1 \mid \hat{y}=1, B)|$

In [ ]:
def parity_gap(y, yhat, group, kind):
    """Return |metric_A - metric_B| for kind in {'demographic', 'equal_opportunity', 'predictive_parity'}.

    TODO(you):
    1. For each group g in ('A', 'B'), compute the requested rate.
       - demographic: mean(yhat[group == g])
       - equal_opportunity: mean(yhat[(group == g) & (y == 1)])
       - predictive_parity: mean(y[(group == g) & (yhat == 1)])
    2. Return the absolute difference.
    """
    # TODO
    return ...

In [ ]:
demo = parity_gap(y, yhat, group, 'demographic')
eo = parity_gap(y, yhat, group, 'equal_opportunity')
pp = parity_gap(y, yhat, group, 'predictive_parity')

print(f'Demographic-parity gap:   {demo:.3f}')
print(f'Equal-opportunity gap:    {eo:.3f}')
print(f'Predictive-parity gap:    {pp:.3f}')

ref = parity_metrics(y, yhat, group)
assert abs(demo - abs(ref['A']['positive_rate'] - ref['B']['positive_rate'])) < 1e-9
assert abs(eo - abs(ref['A']['tpr'] - ref['B']['tpr'])) < 1e-9
assert abs(pp - abs(ref['A']['ppv'] - ref['B']['ppv'])) < 1e-9
print('\nAll asserts pass. Pick the parity that matches the application — you cannot have all three when base rates differ.')

<details>
<summary>Solution</summary>

```python
def parity_gap(y, yhat, group, kind):
    rates = {}
    for g in ('A', 'B'):
        m = group == g
        if kind == 'demographic':
            rates[g] = yhat[m].mean()
        elif kind == 'equal_opportunity':
            sel = m & (y == 1)
            rates[g] = yhat[sel].mean()
        elif kind == 'predictive_parity':
            sel = m & (yhat == 1)
            rates[g] = y[sel].mean()
        else:
            raise ValueError(kind)
    return abs(rates['A'] - rates['B'])
```

Carrying these three gaps on the same dashboard — alongside accuracy and calibration — is the basis of an operational fairness audit. When stakeholders pick a parity to optimise, the others go in the model card as known trade-offs.

</details>

## Recap

- The three parity definitions (demographic, equal opportunity, predictive parity) cannot all hold when base rates differ. Pick one with stakeholders.
- Aggregate calibration can hide per-group gaps. Report reliability by subgroup before showing confidence in the UI.
- Differential privacy buys provable guarantees with predictable accuracy cost — pick epsilon explicitly, track the global budget.
- Recommenders that retrain on their own outputs collapse without diversity injection. A one-line epsilon-greedy is the floor.